# Agent Integration

This notebook connects the knowledge base created in [Knowledge layer](knowledge-layer.ipynb) to a **Foundry Agent** using the **Model Context Protocol (MCP)**. By the end, you'll have:

1. A **project connection** that links your Foundry project to the knowledge base's MCP endpoint
2. An **agent** with an MCP tool that invokes `knowledge_base_retrieve` to answer questions
3. Multi-turn **conversations** that demonstrate knowledge-grounded responses with citations

> **Why two unrelated datasets?** The knowledge base deliberately combines NASA Earth at Night (indexed) with Microsoft Learn (remote MCP). Because the sources don't overlap, the agent's citations make it obvious which source answered each part of a question, so you can see agentic retrieval route work across sources.

## Architecture

![Agent integration: a question goes to a Foundry agent, which calls the knowledge base through an MCP tool and project connection, then answers with cited sources](../docs/images/agent-mcp.svg)

## Prerequisites

- **`knowledge-layer.ipynb` completed** — the knowledge base `foundry-iq-demo-kb` must exist on your search service
- **Microsoft Foundry project** with an LLM deployment (e.g., `gpt-5.4-mini`)
- **Foundry roles**: Foundry User (access models/agents), Foundry Project Manager (create project connections)
- Project's managed identity has **Search Index Data Reader** on the search service
- A `.env` file with `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_PROJECT_RESOURCE_ID`

## Phase 2A: Setup & Configuration

In [ ]:
# Install required packages (uncomment if not already installed)
# %pip install azure-ai-projects>=2.0.0 requests azure-identity python-dotenv

In [ ]:
import os
import requests
from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

# Azure AI Search (from knowledge-layer.ipynb)
search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
# Same API version that azure-search-documents 12.2.0b1 targets in every other notebook
search_api_version = os.environ.get("SEARCH_API_VERSION", "2026-11-01-preview")

# Microsoft Foundry Project
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
project_resource_id = os.environ["FOUNDRY_PROJECT_RESOURCE_ID"]

# Agent model
agent_model = os.environ.get("LLM_DEPLOYMENT", "gpt-5.4-mini")

# Knowledge base name (must match knowledge-layer.ipynb)
KNOWLEDGE_BASE_NAME = "foundry-iq-demo-kb"

# Names for resources created in this notebook
PROJECT_CONNECTION_NAME = "foundry-iq-demo-mcp-connection"
AGENT_NAME = "foundry-iq-demo-agent"

# MCP endpoint for the knowledge base
MCP_ENDPOINT = (
    f"{search_endpoint}/knowledgebases/{KNOWLEDGE_BASE_NAME}"
    f"/mcp?api-version={search_api_version}"
)

credential = DefaultAzureCredential()

print(f"Search endpoint:  {search_endpoint}")
print(f"Project endpoint: {project_endpoint}")
print(f"MCP endpoint:     {MCP_ENDPOINT}")
print(f"Agent model:      {agent_model}")


## Phase 2B: Create Project Connection

A **project connection** of type `RemoteTool` tells Foundry Agent Service how to reach the knowledge base's MCP endpoint. The connection uses the project's **managed identity** to authenticate to Azure AI Search, so no API keys need to be stored.

We create this connection via the Azure Resource Manager REST API.

In [ ]:
# Create a project connection to the knowledge base MCP endpoint
bearer_token_provider = get_bearer_token_provider(
    credential, "https://management.azure.com/.default"
)

headers = {
    "Authorization": f"Bearer {bearer_token_provider()}",
}

connection_url = (
    f"https://management.azure.com{project_resource_id}"
    f"/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview"
)

connection_body = {
    "name": PROJECT_CONNECTION_NAME,
    "type": "Microsoft.MachineLearningServices/workspaces/connections",
    "properties": {
        "authType": "ProjectManagedIdentity",
        "category": "RemoteTool",
        "target": MCP_ENDPOINT,
        "isSharedToAll": True,
        "audience": "https://search.azure.com/",
        "metadata": {"ApiType": "Azure"},
    },
}

response = requests.put(connection_url, headers=headers, json=connection_body)
response.raise_for_status()

print(f"Connection '{PROJECT_CONNECTION_NAME}' created or updated successfully.")
print(f"  Target: {MCP_ENDPOINT}")
print(f"  Auth:   ProjectManagedIdentity")

## Phase 2C: Create Agent with MCP Tool

We create a Foundry Agent that integrates the knowledge base as an **MCP tool**. The agent:

- Uses instructions that explicitly direct it to call the knowledge base for all answers
- Is configured with `knowledge_base_retrieve` as its only allowed MCP tool
- Uses the project connection we just created for secure authentication

> **Note**: We use `allow_preview=True` on the `AIProjectClient` so we can get an agent-scoped OpenAI client via `get_openai_client(agent_name=...)`. This points the client directly at the agent's endpoint — no `extra_body` hacks needed.

In [ ]:
# Create the project client (allow_preview enables agent-scoped OpenAI client)
project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=credential,
    allow_preview=True,
)

# Agent instructions optimized for knowledge retrieval
AGENT_INSTRUCTIONS = """
You are a helpful assistant that must use the knowledge base to answer all questions.
You must never answer from your own knowledge under any circumstances.

Guidelines:
- Always call the knowledge base tool before responding.
- Provide clear, concise answers based on the retrieved content.
- Include citations to source documents using the annotation format.
- If the knowledge base does not contain the answer, respond with "I don't know based on the available sources."
- When information comes from multiple sources, synthesize them into a coherent response.
""".strip()

# Create the MCP tool pointing to the knowledge base
mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=MCP_ENDPOINT,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME,
)

# Create the agent
agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=agent_model,
        instructions=AGENT_INSTRUCTIONS,
        tools=[mcp_kb_tool],
    ),
)

print(f"Agent '{AGENT_NAME}' created successfully.")
print(f"  Model:   {agent_model}")
print(f"  Version: {agent.version}")
print(f"  Tools:   knowledge_base_retrieve (via MCP)")

## Phase 2D: Invoke the Agent

We get an **agent-scoped OpenAI client** using `get_openai_client(agent_name=...)`, which points directly at the agent's endpoint. Multi-turn context is maintained by chaining `previous_response_id`.

When the agent receives a question, it:

1. Calls `knowledge_base_retrieve` via MCP
2. The knowledge base decomposes the query into subqueries
3. Subqueries run against both knowledge sources in parallel
4. Results are reranked and returned to the agent
5. The agent synthesizes a natural-language response with citations

Citation markers such as `【5:1†source】` in the answer come from the agent; the **SOURCES** list below each answer names the page or service they point to.


In [ ]:
import re
from urllib.parse import urlparse

# Get an agent-scoped OpenAI client — points directly at the agent's endpoint
openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
print(f"Agent-scoped OpenAI client ready for: {AGENT_NAME}")


def describe_citation(url: str) -> str:
    """Turn a citation URL into a readable source name."""
    parsed = urlparse(url)
    if parsed.scheme == "mcp":
        return "Knowledge base synthesized answer"
    if "learn.microsoft.com" in parsed.netloc:
        return "Microsoft Learn (MCP)"
    match = re.search(r"/indexes/([^/]+)/docs/([^?]+)", parsed.path)
    if match:
        page = re.search(r"page_(\d+)", match.group(2))
        return f"Earth at Night, page {page.group(1)}" if page else f"{match.group(1)}: {match.group(2)}"
    return url


def display_response(response):
    """Display the agent's answer and a de-duplicated list of the sources it cited."""
    print(f"\nAgent: {response.output_text}")

    sources = []
    for item in response.output:
        if item.type != "message":
            continue
        for content in item.content:
            for ann in getattr(content, "annotations", None) or []:
                if ann.type == "url_citation":
                    sources.append(describe_citation(ann.url))
                elif ann.type == "file_citation":
                    sources.append(ann.filename)

    unique = list(dict.fromkeys(sources))
    print(f"\n{'─' * 40}")
    if unique:
        print(f"SOURCES ({len(unique)}):")
        for name in unique:
            print(f"  - {name}")
    else:
        print("(No citations returned)")


In [ ]:
# Query 1: Cross-source — should retrieve from both NASA data and Microsoft Learn
query1 = (
    "What patterns of light pollution can be observed from satellite imagery, "
    "and what Azure AI Search features support building retrieval systems "
    "over scientific document collections?"
)

print(f"User: {query1}")
print("-" * 80)

response1 = openai_client.responses.create(
    model=agent_model,
    input=query1,
)

display_response(response1)

In [ ]:
# Query 2: Follow-up question — demonstrates conversational memory
query2 = "Can you tell me more about how holiday lights affect the seasonal patterns?"

print(f"User: {query2}")
print("-" * 80)

response2 = openai_client.responses.create(
    model=agent_model,
    input=query2,
    previous_response_id=response1.id,
)

display_response(response2)

In [ ]:
# Query 3: Microsoft Learn–focused question — should primarily hit the MCP source
query3 = "How do I create a knowledge source in Azure AI Search for agentic retrieval?"

print(f"User: {query3}")
print("-" * 80)

response3 = openai_client.responses.create(
    model=agent_model,
    input=query3,
    previous_response_id=response2.id,
)

display_response(response3)

In [ ]:
# Query 4: Out-of-scope question — should respond with "I don't know"
query4 = "What was the score of last night's basketball game?"

print(f"User: {query4}")
print("-" * 80)

response4 = openai_client.responses.create(
    model=agent_model,
    input=query4,
    previous_response_id=response3.id,
)

display_response(response4)

## Phase 2E: Cleanup

Uncomment and run the cells below to delete the agent and project connection.

> **Note**: This does not delete the knowledge base or knowledge sources (created in knowledge-layer.ipynb). Delete those separately if needed.

In [ ]:
# # Delete the agent
# project_client.agents.delete_version(
#     agent_name=agent.name,
#     agent_version=agent.version,
# )
# print(f"Agent '{agent.name}' version '{agent.version}' deleted.")

# # Delete the project connection
# bearer_token_provider = get_bearer_token_provider(
#     credential, "https://management.azure.com/.default"
# )
# headers = {"Authorization": f"Bearer {bearer_token_provider()}"}

# delete_url = (
#     f"https://management.azure.com{project_resource_id}"
#     f"/connections/{PROJECT_CONNECTION_NAME}?api-version=2025-10-01-preview"
# )
# response = requests.delete(delete_url, headers=headers)
# response.raise_for_status()
# print(f"Project connection '{PROJECT_CONNECTION_NAME}' deleted.")

## Summary

In this notebook, you:

1. **Created a project connection** linking the Foundry project to the knowledge base's MCP endpoint using managed identity authentication
2. **Created an agent** with an MCP tool (`knowledge_base_retrieve`) that queries the multi-source knowledge base
3. **Ran multi-turn conversations** demonstrating:
   - Cross-source retrieval (indexed data + live Microsoft Learn docs)
   - Conversational memory (follow-up questions)
   - Source-specific routing (Azure docs question → MCP source)
   - Graceful handling of out-of-scope questions

The agent transparently leverages Foundry IQ's agentic retrieval engine — query planning, parallel search across knowledge sources, semantic reranking, and citation-backed answers — all through a single MCP tool connection.